# Plant vs Non-Plant Gate — Train on Colab (experiment E4)

Final Year Project — Muhammad Asad Bukhari. **Runtime: CPU is enough**
(Runtime -> Change runtime type -> Hardware accelerator -> None).
Run the cells top to bottom: **Runtime -> Run all**. About 30–45 minutes.

**What this notebook does:** trains a lightweight binary classifier
that answers "is this a plant leaf?" — the input-validation gate for
the Plant Disease app. A frozen ImageNet-pretrained MobileNetV2 plus a
small head. Leaf images come from the PlantVillage training split;
non-leaf images come from Caltech 101 (people, animals, vehicles,
buildings, objects — plant categories excluded).

**Why:** the disease model is a closed-set 38-way classifier, so it
must label *any* photo as some disease — including human photos. The
gate runs *before* the disease model and rejects non-leaf images, so
they never receive a diagnosis.

**Outputs:** `plant_gate_mobilenetv2.keras` (~14 MB) and
`results/gate_evaluation.json` (accuracy, precision, recall, F1, ROC
AUC, and the chosen decision threshold).


In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('Devices:', tf.config.list_physical_devices())
print('GPUs:', tf.config.list_physical_devices('GPU'), '(CPU is fine here)')


In [ ]:
!git clone https://github.com/Asadshah-coder/leaf-disease-prediction.git
%cd leaf-disease-prediction
!pip install -q -r requirements.txt
!pip install -q gdown scikit-learn
!ls


In [ ]:
# Dataset 1: PlantVillage leaves (1.7 GB) from Google Drive, file ID 13_IbE48q2Nv3whIDxVf8OV60JzZGVySx
!gdown --id 13_IbE48q2Nv3whIDxVf8OV60JzZGVySx -O "Plant Disease FYP.zip"
!ls -lh "Plant Disease FYP.zip"
!unzip -q "Plant Disease FYP.zip"
!mkdir -p data
!mv "Plant Disease FYP/train" data/train
!mv "Plant Disease FYP/valid" data/valid
!echo "train images:"; find data/train -type f | wc -l
!echo "valid images:"; find data/valid -type f | wc -l
!echo "classes:"; ls data/train | wc -l


In [ ]:
# Dataset 2: Caltech 101 non-plant images (people, animals, vehicles, buildings, objects)
# Fallback is CIFAR-10 (all 10 classes are non-plant) if the Caltech download fails.
import os, subprocess
os.makedirs('data/nonplant_raw', exist_ok=True)
r = subprocess.run(['curl', '-L', '-o', 'caltech-101.zip',
                    'https://data.caltech.edu/records/mzrjq-6wc02/files/caltech-101.zip'])
print('Caltech download exit code:', r.returncode, 'size:', os.path.getsize('caltech-101.zip') if os.path.exists('caltech-101.zip') else 0)
if os.path.exists('caltech-101.zip') and os.path.getsize('caltech-101.zip') > 10_000_000:
    get_ipython().system('unzip -q caltech-101.zip')
    # The zip contains caltech-101/101_ObjectCategories.tar.gz (note the
    # subfolder and the .gz). Extract it wherever it is.
    get_ipython().system('find . -maxdepth 2 -name "101_ObjectCategories.tar.gz" -exec tar -xzf {} \\;')
    print('CALTECH_OK =', os.path.isdir('101_ObjectCategories'))
else:
    print('Caltech download failed — falling back to CIFAR-10')
    get_ipython().system('curl -L -o cifar-10-python.tar.gz https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz')
    get_ipython().system('tar -xzf cifar-10-python.tar.gz')
    print('CIFAR_OK =', os.path.isdir('cifar-10-batches-py'))


In [ ]:
# Build the gate dataset: data/gate/{train,val}/{leaf,nonleaf}
# Leaf: sample from PlantVillage train/ (train split) and valid/ (val split).
# Non-leaf: sample from Caltech 101, excluding plant categories.
import os, random, shutil
random.seed(42)
PLANT_CATEGORIES = {'bonsai', 'joshua-tree', 'lotus', 'strawberry', 'sunflower', 'water-lilly'}

def sample_files(folder, per_class=None, total=None):
    files = []
    for cls in sorted(os.listdir(folder)):
        d = os.path.join(folder, cls)
        if not os.path.isdir(d):
            continue
        fs = [os.path.join(d, f) for f in os.listdir(d)
              if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        random.shuffle(fs)
        files.extend(fs[:per_class] if per_class else fs)
    random.shuffle(files)
    return files[:total] if total else files

def copy_to(files, dest):
    os.makedirs(dest, exist_ok=True)
    for i, f in enumerate(files):
        shutil.copy(f, os.path.join(dest, f'{i:05d}.jpg'))

# Leaf positives
leaf_train = sample_files('data/train', per_class=60)   # 38 x 60 = 2280
leaf_val = sample_files('data/valid', per_class=20)     # 38 x 20 = 760
print('leaf train:', len(leaf_train), 'leaf val:', len(leaf_val))

# Non-leaf negatives
if os.path.isdir('101_ObjectCategories'):
    nonleaf_pool = []
    for cls in sorted(os.listdir('101_ObjectCategories')):
        if cls in PLANT_CATEGORIES or cls.startswith('.'):
            continue
        d = os.path.join('101_ObjectCategories', cls)
        if not os.path.isdir(d):
            continue
        nonleaf_pool.extend([os.path.join(d, f) for f in os.listdir(d)
                             if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    random.shuffle(nonleaf_pool)
    nonleaf_train, nonleaf_val = nonleaf_pool[:len(leaf_train)], nonleaf_pool[len(leaf_train):len(leaf_train)+len(leaf_val)]
    print('non-leaf source: Caltech 101, pool:', len(nonleaf_pool))
else:
    import pickle
    def load_cifar_batch(path):
        with open(path, 'rb') as f:
            return pickle.load(f, encoding='bytes')
    imgs = []
    for b in range(1, 6):
        d = load_cifar_batch(f'cifar-10-batches-py/data_batch_{b}')
        imgs.extend([x.reshape(3, 32, 32).transpose(1, 2, 0) for x in d[b'data']])
    from PIL import Image
    os.makedirs('data/nonplant_raw/cifar', exist_ok=True)
    for i, arr in enumerate(imgs):
        Image.fromarray(arr).save(f'data/nonplant_raw/cifar/{i:05d}.jpg')
    pool = [f'data/nonplant_raw/cifar/{i:05d}.jpg' for i in range(len(imgs))]
    random.shuffle(pool)
    nonleaf_train, nonleaf_val = pool[:len(leaf_train)], pool[len(leaf_train):len(leaf_train)+len(leaf_val)]
    print('non-leaf source: CIFAR-10, pool:', len(pool))

copy_to(leaf_train, 'data/gate/train/leaf')
copy_to(nonleaf_train, 'data/gate/train/nonleaf')
copy_to(leaf_val, 'data/gate/val/leaf')
copy_to(nonleaf_val, 'data/gate/val/nonleaf')
for split in ('train', 'val'):
    for cls in ('leaf', 'nonleaf'):
        print(split, cls, len(os.listdir(f'data/gate/{split}/{cls}')))


In [ ]:
# Load the gate datasets (MobileNetV2 preprocessing: 0-255 -> [-1, 1])
import tensorflow as tf
IMAGE_SIZE = (128, 128)
BATCH = 32

def preprocess(x, y):
    return tf.keras.applications.mobilenet_v2.preprocess_input(x), y

train_ds = tf.keras.utils.image_dataset_from_directory(
    'data/gate/train', labels='inferred', label_mode='binary',
    color_mode='rgb', batch_size=BATCH, image_size=IMAGE_SIZE,
    shuffle=True, seed=42)
val_ds = tf.keras.utils.image_dataset_from_directory(
    'data/gate/val', labels='inferred', label_mode='binary',
    color_mode='rgb', batch_size=BATCH, image_size=IMAGE_SIZE,
    shuffle=False)
print('class names:', train_ds.class_names)  # ['leaf', 'nonleaf'] alphabetically -> leaf=0, nonleaf=1
train_ds = train_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
# NOTE: image_dataset_from_directory sorts classes alphabetically, so
# leaf=0 and nonleaf=1. The model below outputs P(nonleaf); we convert
# to P(leaf) = 1 - output at evaluation time.


In [ ]:
# Build the gate: frozen MobileNetV2 (ImageNet) + small head
import tensorflow as tf
base = tf.keras.applications.MobileNetV2(
    weights='imagenet', include_top=False,
    input_shape=(128, 128, 3))
base.trainable = False
model = tf.keras.Sequential([
    base,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid'),  # P(nonleaf)
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='binary_crossentropy',
              metrics=['accuracy'])
model.summary()


In [ ]:
# Train the gate head — about 15-25 minutes on a Colab CPU runtime.
!mkdir -p results
history = model.fit(train_ds, validation_data=val_ds, epochs=5)
import json
with open('results/gate_history.json', 'w') as f:
    json.dump({k: [float(v) for v in vals] for k, vals in history.history.items()}, f)
print('final train acc:', history.history['accuracy'][-1])
print('final val acc:', history.history['val_accuracy'][-1])


In [ ]:
# Evaluate the gate: accuracy, precision, recall, F1, ROC AUC,
# and the decision threshold (leaf recall >= 99% preferred).
import json, numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_auc_score,
                             roc_curve)
import tensorflow as tf

y_true, y_score = [], []  # y_score = P(leaf)
for x, y in val_ds.unbatch().batch(256):
    p_nonleaf = model.predict(x, verbose=0).flatten()
    y_true.extend((1 - y.numpy()).tolist())       # leaf=1
    y_score.extend((1 - p_nonleaf).tolist())      # P(leaf)
y_true = np.array(y_true)
y_score = np.array(y_score)
auc = roc_auc_score(y_true, y_score)
fpr, tpr, thr = roc_curve(y_true, y_score)
# Threshold: the HIGHEST one that still keeps leaf recall (TPR) >= 0.99
# (roc_curve lists thresholds from high to low, so the first index
# with TPR >= 0.99 is the strictest gate that still lets >= 99% of
# real leaves through). If no threshold reaches it, fall back to the
# threshold with max recall.
if np.any(tpr >= 0.99):
    idx = int(np.argmax(tpr >= 0.99))
else:
    idx = int(np.argmax(tpr))
chosen_thr = float(thr[idx])
print(f'AUC: {auc:.4f}')
print(f'Chosen threshold (leaf recall >= 99%): {chosen_thr:.4f} '
      f'(leaf recall {tpr[idx]:.4f}, non-leaf rejection {1 - fpr[idx]:.4f})')

def report(thr_value, label):
    pred = (y_score >= thr_value).astype(int)
    print(f'--- {label} (threshold {thr_value:.4f}) ---')
    print('accuracy:', round(accuracy_score(y_true, pred), 4))
    print('precision (leaf):', round(precision_score(y_true, pred), 4))
    print('recall (leaf):', round(recall_score(y_true, pred), 4))
    print('F1 (leaf):', round(f1_score(y_true, pred), 4))
    print('confusion [[TN, FP], [FN, TP]] (rows=true nonleaf/leaf):')
    print(confusion_matrix(y_true, pred))
    return {
        'threshold': thr_value,
        'accuracy': float(accuracy_score(y_true, pred)),
        'precision_leaf': float(precision_score(y_true, pred)),
        'recall_leaf': float(recall_score(y_true, pred)),
        'f1_leaf': float(f1_score(y_true, pred)),
        'confusion': confusion_matrix(y_true, pred).tolist(),
    }

results = {
    'auc': float(auc),
    'n_val': int(len(y_true)),
    'at_0.5': report(0.5, 'threshold 0.5'),
    'at_chosen': report(chosen_thr, 'chosen threshold'),
}
with open('results/gate_evaluation.json', 'w') as f:
    json.dump(results, f, indent=2)
print(open('results/gate_evaluation.json').read())


In [ ]:
# Save the gate model + check its size (target: under ~20 MB for the free Streamlit tier)
model.save('plant_gate_mobilenetv2.keras')
!ls -lh plant_gate_mobilenetv2.keras results/gate_evaluation.json


## Done — next steps

1. **Download `plant_gate_mobilenetv2.keras`** from the Files panel
   (left sidebar) — it is the trained gate.
2. **Copy the numbers** from `results/gate_evaluation.json` into the
   experiment log `docs/experiments/E4.md` (accuracy, precision,
   recall, F1, AUC, chosen threshold) — only quote numbers from this run.
3. **Host the model** in the Google Drive folder **FYP-Leaf-Disease**
   (same folder as the disease model) and put its file ID into
   `GATE_DRIVE_FILE_ID` in `input_validation.py` — the app then
   downloads it on first run, exactly like the disease model.
4. **Update `GATE_THRESHOLD`** in `input_validation.py` to the chosen
   threshold from `results/gate_evaluation.json`.
5. **Push** the notebook + experiment log to GitHub; **never** commit
   the `.keras` file or the dataset (both are in `.gitignore`).
6. Re-run the live test: the selfie must now be **rejected**, and the
   6 leaf photos must still **pass**.
